# Model 1 — MediaPipe Rule-Based Squat Detector
Uses MediaPipe Pose to extract landmarks and computes knee/hip angles.
A squat is detected when both knee angles drop below a threshold.
Exports a JSON config (thresholds) — no training data required.

In [10]:
%pip install mediapipe opencv-python numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\gusta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [11]:
import numpy as np
import json
import urllib.request

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

# Download the pose landmarker model
print("Downloading pose landmarker model...")
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task",
    "pose_landmarker.task"
)
print("Done.")

Done.


In [12]:
import mediapipe as mp
import cv2
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# Landmark indices (same as before, just accessed differently)
LEFT_SHOULDER = 11
LEFT_HIP      = 23
LEFT_KNEE     = 25
LEFT_ANKLE    = 27
RIGHT_HIP     = 24
RIGHT_KNEE    = 26
RIGHT_ANKLE   = 28

SQUAT_CONFIG = {
    "knee_angle_threshold": 120,
    "hip_angle_threshold": 130,
    "visibility_threshold": 0.6,
}

def is_squat_frame(landmarks):
    def get(idx):
        p = landmarks[idx]
        return [p.x, p.y], p.visibility

    shoulder_l, v1 = get(LEFT_SHOULDER)
    hip_l,      v2 = get(LEFT_HIP)
    knee_l,     v3 = get(LEFT_KNEE)
    ankle_l,    v4 = get(LEFT_ANKLE)
    hip_r,      v5 = get(RIGHT_HIP)
    knee_r,     v6 = get(RIGHT_KNEE)
    ankle_r,    v7 = get(RIGHT_ANKLE)

    if min(v1,v2,v3,v4,v5,v6,v7) < SQUAT_CONFIG["visibility_threshold"]:
        return False, {}

    knee_angle_l = calculate_angle(hip_l, knee_l, ankle_l)
    knee_angle_r = calculate_angle(hip_r, knee_r, ankle_r)
    hip_angle_l  = calculate_angle(shoulder_l, hip_l, knee_l)

    in_squat = (
        knee_angle_l < SQUAT_CONFIG["knee_angle_threshold"] and
        knee_angle_r < SQUAT_CONFIG["knee_angle_threshold"] and
        hip_angle_l  < SQUAT_CONFIG["hip_angle_threshold"]
    )

    return in_squat, {
        "knee_angle_left":  round(knee_angle_l, 1),
        "knee_angle_right": round(knee_angle_r, 1),
        "hip_angle_left":   round(hip_angle_l, 1),
    }

print("Config and helpers ready.")

Config and helpers ready.


In [13]:
base_options = mp_python.BaseOptions(model_asset_path="pose_landmarker.task")
options = mp_vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.VIDEO,
)

cap = cv2.VideoCapture(0)
squat_count = 0
in_squat = False
timestamp_ms = 0

with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        timestamp_ms += int(1000 / 30)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.pose_landmarks:
            landmarks = result.pose_landmarks[0]
            squat, angles = is_squat_frame(landmarks)

            if squat and not in_squat:
                squat_count += 1
                in_squat = True
            elif not squat:
                in_squat = False

            label = "SQUAT" if squat else "STANDING"
            color = (0, 255, 0) if squat else (0, 0, 255)
            cv2.putText(frame, f"{label}  Reps: {squat_count}", (20, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 2)
            if angles:
                cv2.putText(frame, f"Knee L/R: {angles['knee_angle_left']}/{angles['knee_angle_right']}",
                            (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 1)

        cv2.imshow("Squat Detector - Model 1", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()

In [14]:
# --- Export model config ---
with open("squat_model1_config.json", "w") as f:
    json.dump(SQUAT_CONFIG, f, indent=2)

print("Exported: squat_model1_config.json")

Exported: squat_model1_config.json
